# Amazon Scaping Laptops Data

In [160]:
# install beautifulsoup4 and requests
!pip install beautifulsoup4 requests

In [161]:
# import required libraries
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

In [162]:
# url of the page to be scraped
url ="https://www.amazon.in/s?k=laptops&crid=3O5DJEZW2W1BU&sprefix=lapto%2Caps%2C809&ref=nb_sb_noss_2"

In [163]:
# make header to mimic a browser visit
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

In [164]:
# check if the url is accessible
response = requests.get(url, headers=headers)
if response.status_code == 200:
    print("URL is accessible")
else:
    print("URL is not accessible")    

URL is accessible


In [165]:
# check the response content
print(response.content)

b'<!doctype html><html lang="en-in" class="a-no-js" data-19ax5a9jf="dingo"><!-- sp:feature:head-start -->\n<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>\n<!-- sp:end-feature:head-start -->\n<!-- sp:feature:csm:head-open-part1 -->\n\n<script type=\'text/javascript\'>var ue_t0=ue_t0||+new Date();</script>\n<!-- sp:end-feature:csm:head-open-part1 -->\n<!-- sp:feature:cs-optimization -->\n<meta http-equiv=\'x-dns-prefetch-control\' content=\'on\'>\n<link rel="preconnect" href="https://images-eu.ssl-images-amazon.com" crossorigin>\n<link rel="preconnect" href="https://m.media-amazon.com" crossorigin>\n<!-- sp:end-feature:cs-optimization -->\n<!-- sp:feature:csm:head-open-part2 -->\n<script type=\'text/javascript\'>\nwindow.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;\nif (window.ue_ihb === 1) {\n\nvar ue_csm = window,\n    ue_hob = +new Date();\n(function(d){var e=d.ue=d.ue||{},f=Date.now||function(){return+new Date};e.d=function(b){return f()

In [166]:
# convert the response content to a BeautifulSoup object
soup = BeautifulSoup(response.content, "html.parser")
soup

<!DOCTYPE html>
<html class="a-no-js" data-19ax5a9jf="dingo" lang="en-in"><!-- sp:feature:head-start -->
<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>
<!-- sp:end-feature:head-start -->
<!-- sp:feature:csm:head-open-part1 -->
<script type="text/javascript">var ue_t0=ue_t0||+new Date();</script>
<!-- sp:end-feature:csm:head-open-part1 -->
<!-- sp:feature:cs-optimization -->
<meta content="on" http-equiv="x-dns-prefetch-control"/>
<link crossorigin="" href="https://images-eu.ssl-images-amazon.com" rel="preconnect"/>
<link crossorigin="" href="https://m.media-amazon.com" rel="preconnect"/>
<!-- sp:end-feature:cs-optimization -->
<!-- sp:feature:csm:head-open-part2 -->
<script type="text/javascript">
window.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;
if (window.ue_ihb === 1) {

var ue_csm = window,
    ue_hob = +new Date();
(function(d){var e=d.ue=d.ue||{},f=Date.now||function(){return+new Date};e.d=function(b){return f()-(b?0:d.ue_t0)};e.st

In [167]:
# create the empty lists to store the scraped data
data = []
print("Data cleared")

Data cleared


In [168]:
for page in range(1,50):

    #paramters to be sent in the get request
    params = {
        "k": "laptops",
        "page": page
    }

    # get response from the server
    response = requests.get(url, headers=headers, params=params)

    # convert the response content to a BeautifulSoup object
    soup = BeautifulSoup(response.content, "html.parser")

    # find all the product items on the page using correct selector
    product_containers = soup.find_all("div", class_="s-result-item")

    # extract the required data from each product container
    for container in product_containers:

        # Step 1: Extract title from h2 tag
        h2_tag = container.find("h2")
        if not h2_tag:
            continue
        
        title = h2_tag.text.strip()
        
        # Skip if title is too short or is a header
        if len(title) < 10 or title in ["Results", "Trending now"]:
            continue

        # Step 2: Extract product price
        price_tag = container.find("span", class_="a-price-whole")
        price = price_tag.text.strip() if price_tag else "N/A"

        # Step 3: Extract product brand (first word)
        match = re.search(r"^([A-Za-z]+)", title)
        brand = match.group(1) if match else "Unknown"

        # Step 4 : Extract the ram form the title using regex GB,ram DDR LPDDR
        match = re.search(r"(\d+GB|RAM\s*(\d+GB)?|DDR\d?\s*(\d+GB)?|LPDDR\d?\s*(\d+GB)?)", title, re.IGNORECASE)
        ram = match.group(1) if match else "N/A"

        # Step 5 : Extract the SSD storage from the title using reges
        match = re.search(r"(\d+)\s*(GB|TB)\s*(?:SSD|Storage|HDD)", title, re.IGNORECASE)
        ssd_storage = f"{match.group(1)}{match.group(2)}" if match else "N/A"
        
        #Step 6 : Extract the Color from the title using regex
        match = re.search(r"\b(Black|White|Silver|Gray|Grey|Red|Blue|Green|Yellow|Pink|Purple|Gold|Bronze|Rose Gold|Indigo|Glacier)\b", title, re.IGNORECASE)
        color = match.group(1) if match else "N/A"

        # Step 7: Extract the processor (Intel, AMD, Apple M/A chip, Snapdragon, MediaTek, etc)
        processor = "N/A"
        # Try to match Apple M series (M1, M2, M3, M4, M5, etc.)
        match = re.search(r"Apple\s+M(\d+)", title, re.IGNORECASE)
        if match:
            processor = f"Apple M{match.group(1)}"
        else:
            # Try to match Apple A series (A18, A17, A16, etc.)
            match = re.search(r"Apple\s+A(\d+)", title, re.IGNORECASE)
            if match:
                processor = f"Apple A{match.group(1)}"
            else:
                # Try other processor keywords
                processor_keywords = ["Intel", "AMD", "Snapdragon", "MediaTek", "Celeron"]
                for keyword in processor_keywords:
                    if re.search(rf"\b{keyword}\b", title, re.IGNORECASE):
                        processor = keyword
                        break
        
        # Step 8: Extract the OS from the title using regex
        os = "N/A"
        # Check for MacBook/macOS first (Apple specific)
        if re.search(r"\b(MacBook|macOS|Mac OS)\b", title, re.IGNORECASE):
            os = "MacOS"
        # Check for Apple processors (M or A series means MacOS)
        elif re.search(r"\bApple\s+[MA]\d+\b", title, re.IGNORECASE):
            os = "MacOS"
        # Check for Windows variations
        elif re.search(r"\b(Windows|Win\s*11|Win11|Win\s*10|Win10)\b", title, re.IGNORECASE):
            os = "Windows"
        # Check for other OS keywords
        else:
            OS_keywords = ["Linux", "Chrome OS", "Android", "iOS"]
            for keyword in OS_keywords:
                if re.search(rf"\b{keyword}\b", title, re.IGNORECASE):
                    os = keyword
                    break
            
        # Step 9: Extract the Rating from the container
        rating = "N/A"
        # Try multiple methods to find rating
        # Method 1: Look for i-star span
        rating_span = container.find("span", class_=lambda x: x and "i-star" in str(x).lower() if x else False)
        
        # Method 2: Look for aria-label attribute with rating
        if not rating_span:
            rating_span = container.find("span", {"aria-label": lambda x: x and "out of 5" in str(x).lower() if x else False})
        
        # Method 3: Look for a-icon-star class
        if not rating_span:
            rating_span = container.find("i", class_=lambda x: x and "a-icon-star" in str(x).lower() if x else False)
        
        if rating_span:
            # Try to get rating from aria-label
            aria_label = rating_span.get("aria-label")
            if aria_label:
                match = re.search(r"(\d+\.?\d*)\s*out of 5", aria_label)
                if match:
                    rating = match.group(1)
            else:
                # Try to get from text
                rating_text = rating_span.text.strip() if rating_span else ""
                match = re.search(r"(\d+\.?\d*)", rating_text)
                if match:
                    rating = match.group(1) 


        # Append the extracted data to the list
        data.append({
            "title": title,
            "price": price,
            'brand' : brand,
            'ram' : ram,
            'ssd_storage' : ssd_storage,
            'color' : color,
            'processor' : processor,
            'os' : os,
            'rating' : rating
        })

print("Total items scraped:", len(data))


Total items scraped: 1115


In [169]:
# print the scraped data in a structured format
for item in data:
    print(f"Title: {item['title']}")
    print(f"Price: {item['price']}")
    print(f"Brand: {item['brand']}")
    print(f"RAM: {item['ram']}")
    print(f"SSD Storage: {item['ssd_storage']}")
    print(f"Color: {item['color']}")
    print(f"Processor: {item['processor']}")
    print(f"OS: {item['os']}")
    print(f"Rating: {item['rating']}")
    print("-" * 40)

Title: HP Smartchoice Omnibook 5 OLED (Previously Pavilion), Snapdragon X Processor (16GB LPDDR5x,1TB SSD) 2K OLED,16''/40.6cm, Win11, M365(1yr)*Office24, Glacier Silver, 1.59kg, fb0001QU,Next-Gen AI Laptop
Price: 67,793
Brand: HP
RAM: 16GB
SSD Storage: 1TB
Color: Glacier
Processor: Snapdragon
OS: Windows
Rating: 4.2
----------------------------------------
Title: HP Omnibook 3, Snapdragon X Processor, 45 TOPS (16GB LPDDR5x, 512GB SSD) 2K, WUXGA, IPS, 14''/35.6cm, Win 11, Office 24, Silver, 1.4kg, hz0026QU, FHD IR Camera, 42% Lighter mini GaN Charger, AI Laptop
Price: 73,990
Brand: HP
RAM: 16GB
SSD Storage: 512GB
Color: Silver
Processor: Snapdragon
OS: Windows
Rating: 3.0
----------------------------------------
Title: Acer Smartchoice Aspire One, AMD Ryzen 3-7320U, 8GB LPDDR5 RAM/ 256GB SSD, 14.0"/35.56cm TN HD Display, Win 11 Home, Pure Silver, 1.48KG, A114-43, Thin and Light Laptop
Price: 39,990
Brand: Acer
RAM: 8GB
SSD Storage: 256GB
Color: Silver
Processor: AMD
OS: Windows
Rating:

In [202]:
# make the dataframe from the scraped data
df = pd.DataFrame(data)
df.head()

,title,price,brand,ram,ssd_storage,color,processor,os,rating
0,HP Smartchoice Omnibook 5 OLED (Previously Pav...,"67,793",HP,16GB,1TB,Glacier,Snapdragon,Windows,4.2
1,"HP Omnibook 3, Snapdragon X Processor, 45 TOPS...","73,990",HP,16GB,512GB,Silver,Snapdragon,Windows,3.0
2,"Acer Smartchoice Aspire One, AMD Ryzen 3-7320U...","39,990",Acer,8GB,256GB,Silver,AMD,Windows,5.0
3,"Dell 15 (Previously Inspiron) Laptop, 14th Gen...","44,990",Dell,8GB,N/A,Black,Intel,Windows,4.1
4,"ASUS TUF A15 (2025), AMD Ryzen 7 7445HS,RTX 30...","68,990",ASUS,4GB,512GB,Black,AMD,Windows,4.2


# Data Preprocessing on the scrap data

In [203]:
# check the info of the dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        1115 non-null   object
 1   price        1115 non-null   object
 2   brand        1115 non-null   object
 3   ram          1115 non-null   object
 4   ssd_storage  1115 non-null   object
 5   color        1115 non-null   object
 6   processor    1115 non-null   object
 7   os           1115 non-null   object
 8   rating       1115 non-null   object
dtypes: object(9)
memory usage: 78.5+ KB


In [204]:
# check the shape of the dataframe
df.shape

(1115, 9)

In [205]:
# check the missing values in the dataframe
df.isnull().sum()

title          0
price          0
brand          0
ram            0
ssd_storage    0
color          0
processor      0
os             0
rating         0
dtype: int64

In [206]:
# replace all N/A values with NaN
df.replace("N/A", pd.NA, inplace=True)

In [207]:
# check missing value 
df.isnull().sum()

title            0
price          193
brand            0
ram            317
ssd_storage    387
color          379
processor      306
os             305
rating         264
dtype: int64

In [208]:
#  drop the title column as it is not required for analysis
df.drop("title", axis=1, inplace=True)

In [209]:
df.head()

,price,brand,ram,ssd_storage,color,processor,os,rating
0,"67,793",HP,16GB,1TB,Glacier,Snapdragon,Windows,4.2
1,"73,990",HP,16GB,512GB,Silver,Snapdragon,Windows,3.0
2,"39,990",Acer,8GB,256GB,Silver,AMD,Windows,5.0
3,"44,990",Dell,8GB,<NA>,Black,Intel,Windows,4.1
4,"68,990",ASUS,4GB,512GB,Black,AMD,Windows,4.2


In [210]:
# all columns are first letter title 
df.columns = df.columns.str.title()

In [211]:
df.head()

,Price,Brand,Ram,Ssd_Storage,Color,Processor,Os,Rating
0,"67,793",HP,16GB,1TB,Glacier,Snapdragon,Windows,4.2
1,"73,990",HP,16GB,512GB,Silver,Snapdragon,Windows,3.0
2,"39,990",Acer,8GB,256GB,Silver,AMD,Windows,5.0
3,"44,990",Dell,8GB,<NA>,Black,Intel,Windows,4.1
4,"68,990",ASUS,4GB,512GB,Black,AMD,Windows,4.2


In [212]:
# Rename multiple columns
df = df.rename(columns = {'Ram' : 'RAM', 'Ssd_Storage' : 'SSD_Storage', 'Os' : 'OS',})

In [213]:
df.head()

,Price,Brand,RAM,SSD_Storage,Color,Processor,OS,Rating
0,"67,793",HP,16GB,1TB,Glacier,Snapdragon,Windows,4.2
1,"73,990",HP,16GB,512GB,Silver,Snapdragon,Windows,3.0
2,"39,990",Acer,8GB,256GB,Silver,AMD,Windows,5.0
3,"44,990",Dell,8GB,<NA>,Black,Intel,Windows,4.1
4,"68,990",ASUS,4GB,512GB,Black,AMD,Windows,4.2


In [214]:
df.head()

,Price,Brand,RAM,SSD_Storage,Color,Processor,OS,Rating
0,"67,793",HP,16GB,1TB,Glacier,Snapdragon,Windows,4.2
1,"73,990",HP,16GB,512GB,Silver,Snapdragon,Windows,3.0
2,"39,990",Acer,8GB,256GB,Silver,AMD,Windows,5.0
3,"44,990",Dell,8GB,<NA>,Black,Intel,Windows,4.1
4,"68,990",ASUS,4GB,512GB,Black,AMD,Windows,4.2


In [215]:
# handled the SSD storage column 

# replace the 1TB with 1024GB in the SSD storage column
df['SSD_Storage'] = df['SSD_Storage'].str.replace('1TB', '1024GB', regex=False)

# replace GB with empty string and convert to numeric
df['SSD_Storage'] = df['SSD_Storage'].str.replace('GB', '', regex=False)
df['SSD_Storage'] = pd.to_numeric(df['SSD_Storage'], errors='coerce')
df['SSD_Storage']

0       1024.0
1        512.0
2        256.0
3          NaN
4        512.0
         ...  
1110     512.0
1111     512.0
1112       NaN
1113       NaN
1114       NaN
Name: SSD_Storage, Length: 1115, dtype: float64